In [4]:
import pandas as pd
import numpy as np

# Dosya yolları
min_max_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\son_hal\SLF_imar_tipi_agirliklar2.xlsx'
bina_sayilari_artis_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\new_buildings_2024_2035_1703.xlsx'
abone_artis_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\SLF_YUK_DAGITIMI\dagitilacak_abone_delta.xlsx'
as_is_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\as-is_abone\as_is_abone_sayılari.xlsx'
output_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\SLF_YUK_DAGITIMI\output_abone_dagilimi_kumulatif3.xlsx'

# Tüketim tipleri
tuketim_tipleri = ['MESKEN', 'SANAYI', 'TICARETHANE', 'TARIMSAL_SULAMA', 'AYDINLATMA']

# Monte Carlo ile ağırlık belirleme fonksiyonu
def calculate_random_weights(min_max_df, imar_types, abone_tipi):
    """
    Belirli bir abone tipi için imar tiplerine göre rastgele ağırlıklar hesaplar
    """
    weights = {}
    
    for imar_type in imar_types:
        # Min ve Max sütun adlarını oluştur
        min_col = f"{imar_type} - min"
        max_col = f"{imar_type} - max"
        
        # İlgili abone tipi satırını bul
        abone_row = min_max_df[min_max_df['Category'] == abone_tipi]
        
        if not abone_row.empty:
            min_val = abone_row[min_col].values[0]
            max_val = abone_row[max_col].values[0]
            
            # Rastgele ağırlık hesapla
            random_weight = np.random.uniform(min_val, max_val)
            weights[imar_type] = random_weight
        else:
            # Abone tipi bulunamazsa 0 ağırlık ver
            weights[imar_type] = 0
    
    return weights

def distribute_electric_load(year, min_max_df, bina_sayilari_df, abone_artis_df, num_simulations=100):
    """
    Abone artışlarını dağıtan fonksiyon - Orjinal dağıtım fonksiyonu ile aynı mantık
    """
    # İlçe bazındaki yıllık abone artışlarını al
    tuketim_yillik = abone_artis_df.loc[:, ['ilce'] + tuketim_tipleri]
    
    # Sonuçları saklamak için boş bir DataFrame
    results_df = bina_sayilari_df[['id', 'left', 'top', 'right', 'bottom', 'ilce']].copy()
    for tuketim_tipi in tuketim_tipleri:
        results_df[tuketim_tipi] = 0
    
    imar_types = bina_sayilari_df.columns[6:]
    
    # Monte Carlo Simülasyonu: num_simulations kez çalıştır
    simulation_results = []
    for _ in range(num_simulations):
        temp_results = results_df.copy()
        # HER ABONE TİPİ İÇİN AYRI DÖNGÜ
        for tuketim_tipi in tuketim_tipleri:
            # Bu abone tipi için ağırlıkları hesapla
            weights = calculate_random_weights(min_max_df, imar_types, tuketim_tipi)
            
            # Her ilçe için hesapla
            for ilce in tuketim_yillik['ilce'].unique():
                ilce_mask = bina_sayilari_df['ilce'] == ilce
                ilce_df = bina_sayilari_df.loc[ilce_mask].copy()
                
                # HER HÜCRENİN BU ABONE TİPİ İÇİN AĞIRLIĞINI HESAPLA
                ilce_df[f'weight_{tuketim_tipi}'] = 0
                for imar_type in imar_types:
                    ilce_df[f'weight_{tuketim_tipi}'] += ilce_df[imar_type] * weights[imar_type]
                
                # BU ABONE TİPİ İÇİN NORMALİZASYON
                total_weight = ilce_df[f'weight_{tuketim_tipi}'].sum()
                # Artış miktarını al (- değer de olabilir)
                artis_miktari = tuketim_yillik.loc[tuketim_yillik['ilce'] == ilce, tuketim_tipi].values[0]
                
                # Sadece ARTIŞLARI dağıt, azalışları dağıtma
                if artis_miktari > 0:
                    if total_weight > 0:
                        # Normalleştirme ve dağıtım işlemi
                        ilce_df[tuketim_tipi] = (ilce_df[f'weight_{tuketim_tipi}'] / total_weight) * artis_miktari
                    else:
                        # DÜZELTME: Ağırlık toplamı 0 ise artık eşit dağıtmıyoruz, hiç dağıtmıyoruz
                        ilce_df[tuketim_tipi] = 0
                        print(f"  UYARI: {ilce} ilçesinde {tuketim_tipi} için toplam ağırlık 0. Artış dağıtılmadı.")
                    
                    # Sonuçları geçici tabloya ekle
                    temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
                else:
                    # Azalış durumunda değer atama - azalışlar ayrı işlenecek
                    temp_results.loc[ilce_mask, tuketim_tipi] = 0

        # Simülasyon sonucunu kaydet
        simulation_results.append(temp_results)
    
    # Simülasyonların ortalamasını al
    final_results = pd.concat(simulation_results).groupby(['id', 'left', 'top', 'right', 'bottom', 'ilce']).mean().reset_index()
    return final_results

def apply_decrease(current_df, abone_artis_df, bina_artis_df):
    """
    Abone azalışlarını mevcut durumdan çıkaran fonksiyon
    """
    result_df = current_df.copy()
    tuketim_yillik = abone_artis_df.loc[:, ['ilce'] + tuketim_tipleri]
    imar_types = bina_artis_df.columns[6:]
    
    # Hızlı ID erişimi için sözlük oluştur
    id_to_index = {id_val: i for i, id_val in enumerate(result_df['id'])}
    
    # Her abone tipi için işlem yap
    for tuketim_tipi in tuketim_tipleri:
        # Her ilçe için kontrol et
        for ilce in tuketim_yillik['ilce'].unique():
            artis_miktari = tuketim_yillik.loc[tuketim_yillik['ilce'] == ilce, tuketim_tipi].values[0]
            
            # Sadece azalış durumunda işlem yap
            if artis_miktari < 0:
                azalis_miktari = abs(artis_miktari)
                
                # İlçeye ait hücreleri al
                ilce_cells = result_df[result_df['ilce'] == ilce]
                
                # Bina artışı olmayan hücreleri bul
                no_increase_cells = []
                ilce_bina_artis = bina_artis_df[bina_artis_df['ilce'] == ilce]
                
                for cell_id in ilce_cells['id'].values:
                    cell_artis = ilce_bina_artis[ilce_bina_artis['id'] == cell_id]
                    
                    if not cell_artis.empty:
                        has_increase = False
                        for imar_type in imar_types:
                            if cell_artis[imar_type].sum() > 0:
                                has_increase = True
                                break
                        
                        if not has_increase and cell_id in id_to_index:
                            no_increase_cells.append(cell_id)
                
                # Rastgele karıştır
                np.random.shuffle(no_increase_cells)
                
                # Kalan azaltılacak miktar
                remaining_decrease = azalis_miktari
                
                # Önce bina artışı olmayan hücrelerden azalt
                for cell_id in no_increase_cells:
                    if remaining_decrease <= 0:
                        break
                        
                    idx = id_to_index[cell_id]
                    current_value = result_df.iat[idx, result_df.columns.get_loc(tuketim_tipi)]
                    
                    if current_value > 0:
                        decrease_amount = min(current_value, remaining_decrease)
                        result_df.iat[idx, result_df.columns.get_loc(tuketim_tipi)] -= decrease_amount
                        remaining_decrease -= decrease_amount
                
                # Eğer hala azaltılacak miktar kaldıysa, diğer hücrelerden de azalt
                if remaining_decrease > 0:
                    # İlçedeki diğer hücreleri bul
                    other_cells = [cell_id for cell_id in ilce_cells['id'].values if cell_id not in no_increase_cells and cell_id in id_to_index]
                    np.random.shuffle(other_cells)
                    
                    for cell_id in other_cells:
                        if remaining_decrease <= 0:
                            break
                            
                        idx = id_to_index[cell_id]
                        current_value = result_df.iat[idx, result_df.columns.get_loc(tuketim_tipi)]
                        
                        if current_value > 0:
                            decrease_amount = min(current_value, remaining_decrease)
                            result_df.iat[idx, result_df.columns.get_loc(tuketim_tipi)] -= decrease_amount
                            remaining_decrease -= decrease_amount
    
    return result_df

# Ana fonksiyon: Kümülatif hesaplama
def calculate_cumulative_distribution():
    print("Elektrik yükü dağıtımı kümülatif hesaplama başlıyor...")
    
    # As-is durumu yükle
    try:
        as_is_df = pd.read_excel(as_is_path)
        print(f"As-is durumu yüklendi. Toplam {len(as_is_df)} hücre var.")
    except Exception as e:
        print(f"As-is dosyası yüklenirken hata oluştu: {e}")
        return
    
    # Ağırlık bilgilerini yükle
    try:
        min_max_df = pd.read_excel(min_max_path)
        print("Ağırlık bilgileri yüklendi.")
    except Exception as e:
        print(f"Ağırlık bilgileri yüklenirken hata oluştu: {e}")
        return
    
    # Sonuçları yazmak için Excel writer oluştur
    output_writer = pd.ExcelWriter(output_path, engine='xlsxwriter')
    
    # Yıl aralığı
    years = range(2024, 2036)
    num_simulations = 500  # Monte Carlo simülasyonu kaç kez çalıştırılsın
    
    # Mevcut durumu sakla (kümülatif hesaplama için)
    current_distribution = as_is_df.copy()
    
    # İlk olarak as-is durumu kaydet
    current_distribution.to_excel(output_writer, sheet_name="As-is", index=False)
    print("As-is durumu Excel'e kaydedildi.")
    
    # Her yıl için hesapla
    for year in years:
        print(f"İşleniyor: {year} yılı")
        
        try:
            # O yıla ait verileri yükle
            bina_artis_df = pd.read_excel(bina_sayilari_artis_path, sheet_name=str(year))
            abone_artis_df = pd.read_excel(abone_artis_path, sheet_name=str(year))
            
            # 1. ADIM: Artışları Monte Carlo ile dağıt
            artis_dagilimi = distribute_electric_load(
                year, min_max_df, bina_artis_df, abone_artis_df, num_simulations
            )
            
            # 2. ADIM: Artışları mevcut dağılıma ekle
            result_df = current_distribution.copy()
            
            # ID'lere göre birleştir
            artis_df = artis_dagilimi.copy()
            
            # Artışları ID'ye göre birleştirerek ekle
            result_df = pd.merge(
                result_df,
                artis_df,
                on=['id', 'left', 'top', 'right', 'bottom', 'ilce'],
                how='left',
                suffixes=('', '_artis')
            )
            
            # Artışları mevcut değerlere ekle
            for tuketim_tipi in tuketim_tipleri:
                # NaN değerleri 0 ile doldur
                result_df[f'{tuketim_tipi}_artis'] = result_df[f'{tuketim_tipi}_artis'].fillna(0)
                # Mevcut değer + artış değeri
                result_df[tuketim_tipi] = result_df[tuketim_tipi] + result_df[f'{tuketim_tipi}_artis']
                # Artış sütununu kaldır
                result_df = result_df.drop(f'{tuketim_tipi}_artis', axis=1)
            
            # 3. ADIM: Azalışları uygula
            result_df = apply_decrease(result_df, abone_artis_df, bina_artis_df)
            
            # Sonuçları güncelle ve kaydet
            current_distribution = result_df.copy()
            current_distribution.to_excel(output_writer, sheet_name=str(year), index=False)
            print(f"{year} yılı hesaplandı ve Excel'e kaydedildi.")
            
        except Exception as e:
            print(f"{year} yılı işlenirken hata oluştu: {e}")
            import traceback
            traceback.print_exc()
    
    # Excel dosyasını kapat
    output_writer.close()
    print("Elektrik yükü kümülatif dağıtımı tamamlandı.")

# Programı çalıştır
if __name__ == "__main__":
    calculate_cumulative_distribution()

Elektrik yükü dağıtımı kümülatif hesaplama başlıyor...
As-is durumu yüklendi. Toplam 28149 hücre var.
Ağırlık bilgileri yüklendi.
As-is durumu Excel'e kaydedildi.
İşleniyor: 2024 yılı


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype in

  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde SANAYI için top

C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype in

  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.


...

  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype in

  UYARI: 1 ilçesinde SANAYI için toplam ağırlık 0. Artış dağıtılmadı.
  UYARI: 2 ilçesinde TARIMSAL_SULAMA için toplam ağırlık 0. Artış dağıtılmadı.


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_12708\3217640376.py:91: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0. 0. 0. ... 0. 0. 0.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  temp_results.loc[ilce_mask, tuketim_tipi] = ilce_df[tuketim_tipi]


2035 yılı hesaplandı ve Excel'e kaydedildi.
Elektrik yükü kümülatif dağıtımı tamamlandı.
